Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
# ---- load ----
df2 = pd.read_csv("data/train.csv")
df3 = pd.read_csv("data/score.csv")

print("train:", df2.shape)
print("score:", df3.shape)

train: (15075, 31)
score: (3000, 28)


we have different number of columns, identifying what those columns are

In [4]:
set(df2.columns) - set(df3.columns)

{'claim_count', 'has_safety_program', 'total_loss_amount'}

### EDA. finding relation and effect on target 

Before doing anything i would run a dtypes method to understand with what types of data I am currently working


In [5]:
df2.dtypes

policy_id                               str
insured_id                              str
snapshot_date                           str
policy_effective_date                   str
policy_expiration_date                  str
coverage_type                           str
vehicle_count                         int64
vehicle_avg_age                     float64
driver_count                          int64
driver_avg_age                      float64
years_in_business                   float64
prior_year_mileage_000              float64
business_type                           str
state                                   str
prior_apd_claim_count                 int64
prior_al_claim_count                  int64
prior_loss_amount                   float64
deductible                            int64
coverage_limit_000                    int64
annual_premium                      float64
payment_frequency                       str
risk_score_external                 float64
has_safety_program              

In [6]:
df3.dtypes

policy_id                               str
insured_id                              str
snapshot_date                           str
policy_effective_date                   str
policy_expiration_date                  str
coverage_type                           str
vehicle_count                         int64
vehicle_avg_age                     float64
driver_count                          int64
driver_avg_age                      float64
years_in_business                   float64
prior_year_mileage_000              float64
business_type                           str
state                                   str
prior_apd_claim_count                 int64
prior_al_claim_count                  int64
prior_loss_amount                   float64
deductible                            int64
coverage_limit_000                    int64
annual_premium                      float64
payment_frequency                       str
risk_score_external                 float64
num_heavy_vehicles              

Next step is to understand how many missing data points we have

In [7]:
df2.isnull().sum()

policy_id                               0
insured_id                              0
snapshot_date                           0
policy_effective_date                   0
policy_expiration_date                  0
coverage_type                           0
vehicle_count                           0
vehicle_avg_age                         0
driver_count                            0
driver_avg_age                        603
years_in_business                       0
prior_year_mileage_000                903
business_type                           0
state                                   0
prior_apd_claim_count                   0
prior_al_claim_count                    0
prior_loss_amount                    1208
deductible                              0
coverage_limit_000                      0
annual_premium                          0
payment_frequency                     452
risk_score_external                   752
has_safety_program                      0
num_heavy_vehicles                

In [8]:
df3.isnull().sum()

policy_id                              0
insured_id                             0
snapshot_date                          0
policy_effective_date                  0
policy_expiration_date                 0
coverage_type                          0
vehicle_count                          0
vehicle_avg_age                        0
driver_count                           0
driver_avg_age                       120
years_in_business                      0
prior_year_mileage_000               180
business_type                          0
state                                  0
prior_apd_claim_count                  0
prior_al_claim_count                   0
prior_loss_amount                    240
deductible                             0
coverage_limit_000                     0
annual_premium                         0
payment_frequency                     90
risk_score_external                  150
num_heavy_vehicles                     0
late_payment_count                   210
first_claim_repo

Then I would check duplicates and delete them 

In [9]:
df2[df2.duplicated()]

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,risk_score_external,has_safety_program,num_heavy_vehicles,late_payment_count,claim_count,total_loss_amount,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report
3384,POL-010791,INS-01017,2021-09-12,2021-08-03,2022-08-03,AL,8,6.3,9,39.3,...,72.98,1,1,3.0,0,0.00,NaN,0.00,No Claim,NaN
3740,POL-001140,INS-02828,2021-08-13,2021-06-03,2022-06-03,APD,10,6.0,5,29.0,...,10.57,0,1,0.0,0,0.00,NaN,0.00,No Claim,NaN
4053,POL-009421,INS-01977,2022-07-02,2022-01-31,2023-01-31,AL,6,6.7,9,52.5,...,95.29,0,1,0.0,0,0.00,NaN,0.00,No Claim,NaN
4130,POL-005000,INS-00224,2021-06-29,2021-04-22,2022-04-22,APD,8,2.2,8,37.4,...,2.82,0,0,1.0,0,0.00,NaN,0.00,No Claim,NaN
4207,POL-000266,INS-03340,2021-09-28,2021-05-23,2022-05-23,APD,8,9.9,1,53.1,...,93.21,1,2,1.0,0,0.00,NaN,0.00,No Claim,NaN
5422,POL-008402,INS-03431,2022-08-21,2022-03-23,2023-03-23,AL,11,6.8,8,NaN,...,10.18,1,3,NaN,0,0.00,NaN,0.00,No Claim,NaN
5473,POL-008035,INS-00466,2020-05-23,2020-03-27,2021-03-27,AL,10,2.5,9,33.4,...,91.29,0,2,3.0,0,0.00,NaN,0.00,No Claim,NaN
5523,POL-008286,INS-01411,2022-10-18,2022-06-20,2023-06-20,AL,13,5.3,5,43.5,...,20.96,1,2,1.0,0,0.00,NaN,0.00,No Claim,NaN
5839,POL-014857,INS-00125,2022-04-01,2022-02-08,2023-02-08,AL,13,4.7,6,35.8,...,58.71,1,5,1.0,0,0.00,NaN,0.00,No Claim,NaN
6138,POL-007581,INS-01602,2020-12-09,2020-10-20,2021-10-20,AL,5,5.5,3,44.2,...,59.02,1,0,0.0,0,0.00,NaN,0.00,No Claim,NaN


In [10]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number").fillna(0)
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number").fillna(0)

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"]  = df2["business_type"].astype("category").cat.codes
df2["state"]          = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"]  = df3["business_type"].astype("category").cat.codes
df3["state"]          = df3["state"].astype("category").cat.codes

In [11]:
# ---- target ----
# using binary had-a-claim flag as target
df2["target"] = (df2["claim_count"] > 0).astype(int)
print("positive rate:", round(df2["target"].mean(), 3))

positive rate: 0.123


In [12]:
# ---- features ----
feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium",
    "risk_score_external", "num_heavy_vehicles", "late_payment_count",
]

X = df2[feat_cols]
y = df2["target"]

X.dtypes

coverage_type                int8
vehicle_count               int64
vehicle_avg_age           float64
driver_count                int64
driver_avg_age            float64
years_in_business         float64
prior_year_mileage_000    float64
business_type                int8
state                        int8
prior_apd_claim_count       int64
prior_al_claim_count        int64
prior_loss_amount         float64
deductible                  int64
coverage_limit_000          int64
annual_premium            float64
risk_score_external       float64
num_heavy_vehicles          int64
late_payment_count        float64
dtype: object

In [13]:
# ---- fit model ----

# random split -- not time-based
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tmp = LogisticRegression(max_iter=1000)
tmp.fit(X_train, y_train)

/home/aksel/DataScienceCaseStudy/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [18]:
# ---- evaluate ----
x1 = tmp.predict(X_test)
print("accuracy:", accuracy_score(y_test, x1))
# results look solid

accuracy: 0.8822553897180763


In [20]:
# ---- score new policies ----
X2 = df3[feat_cols]
preds = tmp.predict_proba(X2)

df3["risk_score"] = preds[:, 1]
df3[["policy_id", "risk_score"]].to_csv("predictions.csv")
print("done -- predictions.csv written")


done -- predictions.csv written
